# LGBM

In [1]:
import gc
import os
import lightgbm as lgb
import optuna
import pandas as pd
from sklearn.metrics import roc_auc_score
import pickle

## Функция для оптимизации параметров с помощью optuna

In [2]:
def objective(trial, X_train, y_train, X_val, y_val):
    # Гиперпараметры для LightGBM
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        # Важно: балансируем веса, так как в sampled выборке все еще дисбаланс 1:19
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 19.0),
        "verbose": -1,
        "n_jobs": -1,
    }

    model = lgb.LGBMClassifier(**params)

    # Обучаем с ранней остановкой на валидационной выборке
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

## Настройка логирования

In [3]:
# Настройки путей и логирования
BASE_DIR = "/home/jupyter/project/processed_data"
LOGS_DIR = "/home/jupyter/project/lgbm_optuna_log"
os.makedirs(LOGS_DIR, exist_ok=True)

# Отключаем дефолтные логи Optuna в консоли
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Функция для логирования промежуточных шагов Optuna в файл
def logging_callback(study, trial):
    # Достаем имя набора из кастомных атрибутов исследования
    set_name = study.user_attrs.get("set_name", "unknown")
    log_file_path = os.path.join(LOGS_DIR, f"{set_name}_optimization.log")

    with open(log_file_path, "a") as f:
        f.write(
            f"Trial {trial.number:03d} finished | "
            f"ROC-AUC: {trial.value:.5f} | "
            f"Best ROC-AUC so far: {study.best_value:.5f}\n"
        )

## Подбор параметров модели для каждого набора признаков. Для обучения используем усеченную обучающую выборку

In [10]:
# Наборы признаков
feature_sets = [
    "top300",
    "top500_clean",
    "top500_lgb_clean",
    "top500_magic_meta",
    "top500_micro_engineered",
]

# Загружаем общие таргеты
y_train_sampled = (pd.read_parquet(os.path.join(BASE_DIR, "y_train_sampled.parquet")).iloc[:, 0].values.ravel())
y_val = (pd.read_parquet(os.path.join(BASE_DIR, "y_val.parquet")).iloc[:, 0].values.ravel())

# Словари для сохранения результатов
best_params_per_set = {}
best_scores_per_set = {}

In [ ]:
# Главный цикл по наборам признаков
for folder in feature_sets:
    print(f"\n==================================================")
    print(f" НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: {folder}")
    print(f"==================================================")

    # Подгружаем только нужные матрицы признаков для Optuna
    x_train_sampled_path = os.path.join(BASE_DIR, folder, f"X_train_sampled_{folder}.parquet")
    x_val_path = os.path.join(BASE_DIR, folder, f"X_val_{folder}.parquet")

    X_train_sampled = pd.read_parquet(x_train_sampled_path)
    X_val = pd.read_parquet(x_val_path)

    # Инициализируем исследование Optuna
    study = optuna.create_study(direction="maximize")

    # Сохраняем имя набора внутрь study, чтобы callback знал, куда писать логи
    study.set_user_attr("set_name", folder)

    # Чистим старый лог-файл, если он остался от прошлых запусков
    log_file_path = os.path.join(LOGS_DIR, f"{folder}_optimization.log")
    if os.path.exists(log_file_path):
        os.remove(log_file_path)

    # Запуск оптимизации
    study.optimize(
        lambda trial: objective(
            trial, X_train_sampled, y_train_sampled, X_val, y_val
        ),
        n_trials=50,  # Задайте нужное количество итераций
        timeout=1800,  # Ограничение 30 минут на один датасет
        callbacks=[logging_callback]
    )

    # Сохраняем лучшие результаты
    best_params_per_set[folder] = study.best_params
    best_scores_per_set[folder] = study.best_value

    # Выводим краткие итоги в консоль
    print(f"\n УСПЕШНО ЗАВЕРШЕНО ДЛЯ: {folder}")
    print(f" Лучший ROC-AUC на валидации: {study.best_value:.5f}")
    print(f" Подробный лог сохранен в: {log_file_path}")
    print(f" Лучшие параметры:")
    for param, value in study.best_params.items():
        print(f"   -> {param}: {value}")

    # Очистка памяти перед следующей итерацией
    del X_train_sampled, X_val
    gc.collect();

print("\n\n==================================================")
print(" ОПТИМИЗАЦИЯ ВСЕХ НАБОРОВ ЗАВЕРШЕНА!")
print("==================================================")


 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top300

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top300
 Лучший ROC-AUC на валидации: 0.64964
 Подробный лог сохранен в: /home/jupyter/project/lgbm_optuna_log/top300_optimization.log
 Лучшие параметры:
   -> n_estimators: 425
   -> learning_rate: 0.01346628739270584
   -> num_leaves: 45
   -> max_depth: 12
   -> min_child_samples: 318
   -> subsample: 0.7073686501858086
   -> colsample_bytree: 0.9152416860034637
   -> reg_alpha: 0.1439325628381447
   -> reg_lambda: 0.00046504141948648366
   -> scale_pos_weight: 3.799140505629563

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_clean

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_clean
 Лучший ROC-AUC на валидации: 0.65847
 Подробный лог сохранен в: /home/jupyter/project/lgbm_optuna_log/top500_clean_optimization.log
 Лучшие параметры:
   -> n_estimators: 112
   -> learning_rate: 0.0354205133929837
   -> num_leaves: 195
   -> max_depth: 10
   -> min_child_samples: 199
   -> subsample: 0.7549520679957513
   -> colsample_bytree: 0.5003639154278724
   -> r

## Результаты оптимизации с обучением на усеченной обучающей выборке

In [13]:
# Выводим финальную сравнительную табличку в консоль
for folder in feature_sets:
    print(f"Dataset: {folder:<25} | Best ROC-AUC: {best_scores_per_set[folder]:.5f}")

Dataset: top300                    | Best ROC-AUC: 0.64964
Dataset: top500_clean              | Best ROC-AUC: 0.65847
Dataset: top500_lgb_clean          | Best ROC-AUC: 0.66213
Dataset: top500_magic_meta         | Best ROC-AUC: 0.65056
Dataset: top500_micro_engineered   | Best ROC-AUC: 0.65538


## Обучение моделей с лучшими параметрами на полной обучающей выборке и финальная проверка на валидации

In [14]:
# Настройки путей
BASE_DIR = "/home/jupyter/project/processed_data"
MODELS_DIR = "/home/jupyter/project/lgbm_models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Общие таргеты
y_train_full = (pd.read_parquet(os.path.join(BASE_DIR, "y_train_full.parquet")).iloc[:, 0].values.ravel())
y_val = (pd.read_parquet(os.path.join(BASE_DIR, "y_val.parquet")).iloc[:, 0].values.ravel())

# Словари для хранения результатов финального тестирования
final_val_scores = {}

In [17]:
# Точный коэффициент изменения дисбаланса:
# Нулей стало больше примерно в 5 раз (19 частей превратились в 99 частей)
DISBALANCE_COEF = 99.0 / 19.0

for folder, params in best_params_per_set.items():
    print(f"\n==================================================")
    print(f" ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: {folder}")
    print(f"==================================================")

    X_train_full = pd.read_parquet(os.path.join(BASE_DIR, folder, f"X_train_full_{folder}.parquet"))
    X_val = pd.read_parquet(os.path.join(BASE_DIR, folder, f"X_val_{folder}.parquet"))

    final_params = params.copy()
    final_params["objective"] = "binary"
    final_params["metric"] = "auc"
    final_params["verbose"] = -1
    final_params["n_jobs"] = -1

    # Корректируем вес: увеличиваем его пропорционально наплыву нулей
    final_params["scale_pos_weight"] = min(
        params["scale_pos_weight"] * DISBALANCE_COEF, 99.0
    )

    # Корректируем минимальный размер листа: данных стало в 4 раза больше
    # Увеличиваем порог, чтобы шумные группы нулей не дробили деревья
    final_params["min_child_samples"] = int(params["min_child_samples"] * 3)

    # Сбрасываем n_estimators на максимум, так как мы будем использовать early_stopping
    final_params["n_estimators"] = 3000

    final_model = lgb.LGBMClassifier(**final_params)

    # Обучаем с контролем переобучения на валидационной выборке
    final_model.fit(
        X_train_full,
        y_train_full,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50, verbose=False
            )  # Остановит обучение, если метрика начнет падать
        ],
    )

    # Проверяем итоговый скор
    val_preds = final_model.predict_proba(X_val)[:, 1]
    current_score = roc_auc_score(y_val, val_preds)
    final_val_scores[folder] = current_score

    print(f"Итоговый ROC-AUC после калибровки: {current_score:.5f}")

    # Сохраняем откалиброванную модель
    model_filename = os.path.join(
        MODELS_DIR, f"lgb_{folder}_{current_score:.5f}.pkl"
    )
    with open(model_filename, "wb") as f:
        pickle.dump(final_model, f)

    del X_train_full, X_val, final_model
    gc.collect();


 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top300
Итоговый ROC-AUC после калибровки: 0.64026

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_clean
Итоговый ROC-AUC после калибровки: 0.63126

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_lgb_clean
Итоговый ROC-AUC после калибровки: 0.65509

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_magic_meta
Итоговый ROC-AUC после калибровки: 0.64029

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_micro_engineered
Итоговый ROC-AUC после калибровки: 0.63091


 СРАВНЕНИЕ НОВЫХ РЕЗУЛЬТАТОВ:


## Результаты (стало сильно хуже, кажется, параметры не подходят под полную обучающую выборку)

In [18]:
for folder in best_params_per_set.keys():
    print(
        f"Dataset: {folder:<25} | New Final ROC-AUC: {final_val_scores[folder]:.5f}"
    )

Dataset: top300                    | New Final ROC-AUC: 0.64026
Dataset: top500_clean              | New Final ROC-AUC: 0.63126
Dataset: top500_lgb_clean          | New Final ROC-AUC: 0.65509
Dataset: top500_magic_meta         | New Final ROC-AUC: 0.64029
Dataset: top500_micro_engineered   | New Final ROC-AUC: 0.63091


## Подбор параметров модели для каждого набора признаков. Для обучения сразу используем полную обучающую выборку

In [6]:
# Наборы признаков
feature_sets = [
    "top300",
    "top500_clean",
    "top500_lgb_clean",
    "top500_magic_meta",
    "top500_micro_engineered",
]

# Загружаем общие таргеты
y_train_full = (pd.read_parquet(os.path.join(BASE_DIR, "y_train_full.parquet")).iloc[:, 0].values.ravel())
y_val = (pd.read_parquet(os.path.join(BASE_DIR, "y_val.parquet")).iloc[:, 0].values.ravel())

# Словари для сохранения результатов
best_params_per_set = {}
best_scores_per_set = {}

In [ ]:
# Главный цикл по наборам признаков
for folder in feature_sets:
    print(f"\n==================================================")
    print(f" НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: {folder}")
    print(f"==================================================")

    # Подгружаем только нужные матрицы признаков для Optuna
    x_train_full_path = os.path.join(BASE_DIR, folder, f"X_train_full_{folder}.parquet")
    x_val_path = os.path.join(BASE_DIR, folder, f"X_val_{folder}.parquet")

    X_train_full = pd.read_parquet(x_train_full_path)
    X_val = pd.read_parquet(x_val_path)

    # Инициализируем исследование Optuna
    study = optuna.create_study(direction="maximize")

    # Сохраняем имя набора внутрь study, чтобы callback знал, куда писать логи
    study.set_user_attr("set_name", folder)

    # Чистим старый лог-файл, если он остался от прошлых запусков
    log_file_path = os.path.join(LOGS_DIR, f"{folder}_optimization.log")
    if os.path.exists(log_file_path):
        os.remove(log_file_path)

    # Запуск оптимизации
    study.optimize(
        lambda trial: objective(
            trial, X_train_full, y_train_full, X_val, y_val
        ),
        n_trials=50,  # Задайте нужное количество итераций
        timeout=1800,  # Ограничение 30 минут на один датасет
        callbacks=[logging_callback]
    )

    # Сохраняем лучшие результаты
    best_params_per_set[folder] = study.best_params
    best_scores_per_set[folder] = study.best_value

    # Выводим краткие итоги в консоль
    print(f"\n УСПЕШНО ЗАВЕРШЕНО ДЛЯ: {folder}")
    print(f" Лучший ROC-AUC на валидации: {study.best_value:.5f}")
    print(f" Подробный лог сохранен в: {log_file_path}")
    print(f" Лучшие параметры:")
    for param, value in study.best_params.items():
        print(f"   -> {param}: {value}")

    # Очистка памяти перед следующей итерацией
    del X_train_full, X_val
    gc.collect();

print("\n\n==================================================")
print(" ОПТИМИЗАЦИЯ ВСЕХ НАБОРОВ ЗАВЕРШЕНА!")
print("==================================================")


 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top300

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top300
 Лучший ROC-AUC на валидации: 0.65534
 Подробный лог сохранен в: /home/jupyter/project/lgbm_optuna_log/top300_optimization.log
 Лучшие параметры:
   -> n_estimators: 481
   -> learning_rate: 0.017140635098750034
   -> num_leaves: 189
   -> max_depth: 8
   -> min_child_samples: 344
   -> subsample: 0.8869685736004087
   -> colsample_bytree: 0.5527278559393881
   -> reg_alpha: 0.46858432602292316
   -> reg_lambda: 2.120349713736941e-08
   -> scale_pos_weight: 4.255896472448347

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_clean

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_clean
 Лучший ROC-AUC на валидации: 0.65543
 Подробный лог сохранен в: /home/jupyter/project/lgbm_optuna_log/top500_clean_optimization.log
 Лучшие параметры:
   -> n_estimators: 859
   -> learning_rate: 0.025212849930114602
   -> num_leaves: 58
   -> max_depth: 11
   -> min_child_samples: 488
   -> subsample: 0.7222564146616821
   -> colsample_bytree: 0.5840056697507177
   ->

## Результаты оптимизации с обучением на полной обучающей выборке

In [8]:
# Выводим финальную сравнительную табличку в консоль
for folder in feature_sets:
    print(f"Dataset: {folder:<25} | Best ROC-AUC: {best_scores_per_set[folder]:.5f}")

Dataset: top300                    | Best ROC-AUC: 0.65534
Dataset: top500_clean              | Best ROC-AUC: 0.65543
Dataset: top500_lgb_clean          | Best ROC-AUC: 0.65788
Dataset: top500_magic_meta         | Best ROC-AUC: 0.65288
Dataset: top500_micro_engineered   | Best ROC-AUC: 0.65105


## Финальное обучение моделей с лучшими параметрами на полной обучающей выборке и их сохранение

In [11]:
# Настройки путей
BASE_DIR = "/home/jupyter/project/processed_data"
MODELS_DIR = "/home/jupyter/project/lgbm_models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Загружаем общие таргеты
y_train_full = (pd.read_parquet(os.path.join(BASE_DIR, "y_train_full.parquet")).iloc[:, 0].values.ravel())
y_val = (pd.read_parquet(os.path.join(BASE_DIR, "y_val.parquet")).iloc[:, 0].values.ravel())

final_val_scores = {}

In [12]:
for folder, params in best_params_per_set.items():
    print(f"\n==================================================")
    print(f" ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: {folder}")
    print(f"==================================================")
    
    # Загружаем полные матрицы признаков
    X_train_full = pd.read_parquet(os.path.join(BASE_DIR, folder, f"X_train_full_{folder}.parquet"))
    X_val = pd.read_parquet(os.path.join(BASE_DIR, folder, f"X_val_{folder}.parquet"))

    # Собираем параметры
    final_params = params.copy()
    final_params["objective"] = "binary"
    final_params["metric"] = "auc"
    final_params["verbose"] = -1
    final_params["n_jobs"] = -1

    # Обучаем финальную модель
    final_model = lgb.LGBMClassifier(**final_params)

    # Обучаем с ранней остановкой — она поймает точную лучшую итерацию в пределах этого n_estimators
    final_model.fit(
        X_train_full,
        y_train_full,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    # Проверяем итоговое качество на валидации
    val_preds = final_model.predict_proba(X_val)[:, 1]
    current_score = roc_auc_score(y_val, val_preds)
    final_val_scores[folder] = current_score

    print(f"Проверочный ROC-AUC на валидации: {current_score:.5f}")

    # Сохраняем модель
    model_filename = os.path.join(MODELS_DIR, f"lgb_{folder}_{current_score:.5f}.pkl")
    with open(model_filename, "wb") as f:
        pickle.dump(final_model, f)
    print(f"Модель успешно сохранена: {model_filename}")

    # Очищаем оперативную память
    del X_train_full, X_val, final_model
    gc.collect()

print("\n\n==================================================")
print(" ОБУЧЕНИЕ И ИМПОРТ ВСЕХ МОДЕЛЕЙ ЗАВЕРШЕНЫ!")
print("==================================================")


 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top300
Проверочный ROC-AUC на валидации: 0.65534
Модель успешно сохранена: /home/jupyter/project/lgbm_models/lgb_top300_0.65534.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_clean
Проверочный ROC-AUC на валидации: 0.65543
Модель успешно сохранена: /home/jupyter/project/lgbm_models/lgb_top500_clean_0.65543.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_lgb_clean
Проверочный ROC-AUC на валидации: 0.65788
Модель успешно сохранена: /home/jupyter/project/lgbm_models/lgb_top500_lgb_clean_0.65788.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_magic_meta
Проверочный ROC-AUC на валидации: 0.65288
Модель успешно сохранена: /home/jupyter/project/lgbm_models/lgb_top500_magic_meta_0.65288.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_micro_engineered
Проверочный ROC-AUC на валидации: 0.65105
Модель успешно сохранена: /home/jupyter/project/lgbm_models/lgb_top500_micro_engineered_0.65105.pkl


 ОБУЧЕНИЕ И ИМПОРТ ВСЕХ МОДЕЛЕЙ ЗАВЕРШЕНЫ!
